In [1]:
import re
from datasets import load_dataset
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

In [2]:
def download_dataset():
    story_dataset = load_dataset("cfahlgren1/tinystories-gpt4-clean")
    print(story_dataset)
    print(story_dataset["train"][0])
    return story_dataset

story_dataset = download_dataset()

README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

tinystories_gpt4_clean.parquet: reconstructing file:   0%|          |  0.00B /  673MB            

tinystories_gpt4_clean.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2732634 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2732634
    })
})
{'text': 'Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.\nJack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.\nOnce the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!\nThe more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that.'}


In [3]:
def save_stories_to_file(story_dataset, story_file_path):
    Path("data").mkdir(exist_ok=True)

    with story_file_path.open("w", encoding="utf-8") as story_file:
        for story_number, story_item in enumerate(story_dataset["train"]):
            story_text = story_item["text"]
            story_file.write(story_text)
            story_file.write("\n<END_STORY>\n")

            if (story_number + 1) % 100000 == 0:
                print(f"Processed {story_number + 1} stories")

    print(f"Done! Text saved to: {story_file_path}")

story_file_path = Path("data/tinystories_all.txt")
save_stories_to_file(story_dataset, story_file_path)

Processed 100000 stories
Processed 200000 stories
Processed 300000 stories
Processed 400000 stories
Processed 500000 stories
Processed 600000 stories
Processed 700000 stories
Processed 800000 stories
Processed 900000 stories
Processed 1000000 stories
Processed 1100000 stories
Processed 1200000 stories
Processed 1300000 stories
Processed 1400000 stories
Processed 1500000 stories
Processed 1600000 stories
Processed 1700000 stories
Processed 1800000 stories
Processed 1900000 stories
Processed 2000000 stories
Processed 2100000 stories
Processed 2200000 stories
Processed 2300000 stories
Processed 2400000 stories
Processed 2500000 stories
Processed 2600000 stories
Processed 2700000 stories
Done! Text saved to: data/tinystories_all.txt


In [4]:
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as story_file:
        return story_file.read()

all_story_text = read_text_file("data/tinystories_all.txt")

In [47]:
raw_words = []
raw_words.extend(all_story_text[:1999999].split(" "))
raw_words.append("\n")

In [48]:
def split_punctuation(raw_words):
    split_tokens = []
    for raw_word in raw_words:
        raw_word = raw_word.lower()
        raw_word = re.sub(r"([^a-z0-9\s])", r" \1 ", raw_word)
        split_tokens.extend(raw_word.split())
    return split_tokens

token_list = split_punctuation(raw_words)

In [49]:
def keep_only_letters(token_list):
    letters_only_words = []
    for token in token_list:
        token = token.lower()
        token = re.sub(r"[^a-z]", "", token)
        if token:
            letters_only_words.append(token)
    return letters_only_words

letters_only_words = keep_only_letters(token_list)

In [50]:
unique_sorted_words = sorted(list(set(letters_only_words)))
unique_words_string = " ".join(unique_sorted_words)
print(len(unique_words_string))
unique_words_string

34574


'a able about above accept accepted accident accidentally accidently accidents accomplished ached achieve achieved achievements achoo acorn across act acted acting action actions activity actors actually add added adding adds admire admired admiring adorable adult adults advantage adventure adventures adventurous advice aeroplane afar afloat afraid after afternoon again against age ago agree agreed agreement agrees ah ahead ahh ahhh ahoy aid aiden aim aimed aims air airplane airport aisle alarm alert alex alice alien alive all allergy alligator allow allowed allowing almost alone along alongside already alright also although always am amanda amazed amazement amazing ambitions ambitious ambulance amongst amount amusement amy an anchor ancient and andvery andy angel angels angles angry animal animals ann anna annabelle annie annoyed another answer answered answers ant anthony ants anxious any anymore anyone anything anytime anyway anywhere apart apologised apologize apologized appear app

In [51]:
def build_probability_matrix(token_list):
    vocab_list = list(dict.fromkeys(token_list))
    vocab_size = len(vocab_list)
    word_counts = [token_list.count(word) for word in vocab_list]

    probability_matrix = np.zeros((vocab_size, vocab_size))

    for position in range(len(token_list) - 1):
        current_index = vocab_list.index(token_list[position])
        next_index = vocab_list.index(token_list[position + 1])
        probability_matrix[current_index][next_index] += 1

    for row in range(vocab_size):
        probability_matrix[row] = probability_matrix[row] / word_counts[row]

    return vocab_list, vocab_size, word_counts, probability_matrix

vocab_list, vocab_size, word_counts, probability_matrix = build_probability_matrix(token_list)

In [52]:
probability_table = pd.DataFrame(probability_matrix, index=vocab_list, columns=vocab_list)
probability_table.head(10)

,once,there,was,a,little,boy,named,jack,.,he,...,exhausted,proudest,lung,staff,villagers,politeest,brand,energized,shares,su
once,0.000000,0.044994,0.000634,0.000000,0.000000,0.000000,0.000000,0.000000,0.001267,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
there,0.000487,0.000000,0.737329,0.000487,0.000000,0.000000,0.000000,0.000000,0.044834,0.000487,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
was,0.000213,0.001807,0.000000,0.222045,0.000000,0.000000,0.000213,0.000213,0.006909,0.000000,...,0.000106,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
a,0.000000,0.000000,0.000000,0.000000,0.096648,0.013248,0.000000,0.000000,0.000217,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000072,0.0,0.0,0.0
little,0.000000,0.000000,0.000000,0.000000,0.000000,0.217586,0.000000,0.000497,0.001490,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
boy,0.000000,0.000000,0.065050,0.000000,0.000000,0.000000,0.566703,0.000000,0.066152,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
named,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004692,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
jack,0.000000,0.000000,0.190789,0.013158,0.000000,0.000000,0.000000,0.000000,0.085526,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
.,0.000150,0.001800,0.000000,0.004874,0.000150,0.000000,0.000000,0.001325,0.000200,0.109448,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
he,0.000000,0.000000,0.108177,0.000000,0.000000,0.000000,0.000402,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [53]:
probability_table["boy"]["little"]

np.float64(0.21758569299552907)

In [54]:
def build_word_to_index(token_list):
    word_to_index = {}
    for index, word in enumerate(sorted(list(set(token_list)))):
        word_to_index[word] = index
    return word_to_index

word_to_index = build_word_to_index(token_list)
word_to_index["a"]

23

In [55]:
def build_index_to_word(token_list):
    index_to_word = {}
    for index, word in enumerate(sorted(list(set(token_list)))):
        index_to_word[index] = word
    return index_to_word

index_to_word = build_index_to_word(token_list)
index_to_word[4]

'-'

In [56]:
def data_and_labels():
    xs = []
    ys = []
    for i in range(len(token_list) - 11):
        xs.append(token_list[i:i+11])
        ys.append(token_list[i+11])
    return xs, ys

xs, ys = data_and_labels()

In [57]:
xs[10],[ys[10]]

(['was',
  'only',
  'three',
  'years',
  'old',
  'and',
  'had',
  'lots',
  'of',
  'things',
  'he'],
 ['wanted'])

In [58]:
for i in range(len(ys)):
    xs[i] = [word_to_index[word] for word in xs[i]]
    ys[i] = word_to_index[ys[i]]


In [59]:
xs[1],[ys[1]]

([4313, 4704, 23, 2341, 477, 2647, 2086, 5, 1844, 4704, 2777], [4339])

In [60]:
xs = np.array(xs)
ys = np.array(ys)
SEQ_LEN = 11
vocab_size = len(word_to_index)
print(xs.shape, ys.shape)

(490985, 11) (490985,)


In [61]:
from tensorflow.keras import layers

class WordAndPosition(layers.Layer):
    def __init__(self, vocab_size, size):
        super().__init__()
        self.word_layer = layers.Embedding(vocab_size, size)
        self.position_layer = layers.Embedding(SEQ_LEN, size)

    def call(self, x):
        positions = tf.range(SEQ_LEN)
        return self.word_layer(x) + self.position_layer(positions)

def build_model(vocab_size, size=64, blocks=2):
    model_input = layers.Input((SEQ_LEN,), dtype="int32")
    x = WordAndPosition(vocab_size, size)(model_input)

    for _ in range(blocks):
        look_back = layers.MultiHeadAttention(4, size // 4)(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + look_back)
        think = layers.Dense(size * 4, activation="relu")(x)
        think = layers.Dense(size)(think)
        x = layers.LayerNormalization()(x + think)

    last_word = layers.Lambda(lambda t: t[:, -1, :])(x)
    model_output = layers.Dense(vocab_size)(last_word)
    return tf.keras.Model(model_input, model_output)

gpt_model = build_model(vocab_size)
gpt_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
gpt_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_and_position_2 │ (None, 11, 64)    │    317,376 │ input_layer_2[0]… │
│ (WordAndPosition)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 11, 64)    │     16,640 │ word_and_positio… │
│ (MultiHeadAttentio… │                   │            │ word_and_positio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 11, 64)    │          0 │ word_and_positio… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 11, 64)    │        128 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 11, 256)   │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 11, 64)    │     16,448 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 11, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 11, 64)    │        128 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 11, 64)    │     16,640 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 11, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 11, 64)    │        128 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 11, 256)   │     16,640 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 11, 64)    │     16,448 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 11, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 11, 64)    │        128 │ add_11[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 64)        │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 4948)      │    321,620 │ lambda_2[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 738,964 (2.82 MB)

 Trainable params: 738,964 (2.82 MB)

 Non-trainable params: 0 (0.00 B)

In [62]:
gpt_model.fit(xs, ys, batch_size=64, epochs=20)

Epoch 1/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 50s 5ms/step - loss: 3.4707
Epoch 2/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.9334
Epoch 3/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.7576
Epoch 4/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 38s 5ms/step - loss: 2.6464
Epoch 5/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - loss: 2.5672
Epoch 6/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.5034
Epoch 7/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.4531
Epoch 8/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.4101
Epoch 9/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.3746
Epoch 10/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - loss: 2.3417
Epoch 11/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.3138
Epoch 12/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 36s 5ms/step - loss: 2.2884
Epoch 13/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.2647
Epoch 14/20
7672/7672 ━━━━━━━━━━━━━━━━━━━━ 37s 5ms/step - loss: 2.2442
Epoch 15/20
767

In [64]:
def write_story(start_numbers, how_many_words=30):
    numbers = list(start_numbers)

    for _ in range(how_many_words):
        last_words = np.array([numbers[-SEQ_LEN:]])
        scores = gpt_model.predict(last_words, verbose=0)[0]
        chances = np.exp(scores - scores.max())
        chances = chances / chances.sum()
        numbers.append(np.random.choice(len(chances), p=chances))

    return " ".join(index_to_word[n] for n in numbers)


def prompt(text="once there was a little boy named jack . he", how_many_words=100):
    words = text.lower().split()

    unknown_words = [word for word in words if word not in word_to_index]
    if unknown_words:
        print("Words the model does not know:", unknown_words)
        words = [word for word in words if word in word_to_index]

    start_numbers = [word_to_index[word] for word in words]
    start_numbers = start_numbers[-SEQ_LEN:]

    while len(start_numbers) < SEQ_LEN:
        start_numbers.insert(0, word_to_index["."])

    return write_story(start_numbers, how_many_words)


prompt("what is a boy and a girl doing in the story ?")

"is a boy and a girl doing in the story ? and cat asked her about the bug . the others got angry and lonely . they saw lily ' s strings . they saw fin looking sad . the intelligent dolphin heard tom decides . he came jack saw he island and swam to his pond . he had big buttons and shoes , head and splashed all day long , but sometimes people were very big and they were always popular when they were done . < end _ story > once upon a time , there was a nice boy named tom . tom was a miserable boy"